# XGBoost Two-Stage Tuning - excluding always-100%-recovery sites

This notebook mirrors `xgboost_classification_regression_tuning.ipynb` but runs Optuna tuning for both stages on the subset of the aggregated data that **excludes fulfillment sites whose recovery rate is consistently 1.0**. The goal is to find hyperparameters tailored to the variable-recovery population, separate from the canonical tuned models.

Tuning conventions follow the project standard: by-year split (`year < 2025` train, `year == 2025` test), Stage 1 = `XGBClassifier` for `P(prob_recovered > 0)`, Stage 2 = `XGBRegressor` in logit space on non-zero rows with the site-GL baseline features and GL/site/global priors. Tuned models are saved to S3 with a `_excluding_full_recovery` suffix so the canonical artifacts remain untouched.

# XGBoost Two-Stage Model — Hyperparameter Tuning with Optuna

This notebook performs Optuna hyperparameter tuning on the two-stage model defined in `xgboost_classification_regression.ipynb`:

- **Stage 1**: `XGBClassifier` predicting `P(prob_recovered > 0)`
- **Stage 2**: `XGBRegressor` predicting `E(prob_recovered | prob_recovered > 0)` in logit space

The final prediction is the product `p_nonzero * e_rate`.

Designed to run on AWS with S3 access.

In [ ]:
!pip install shap
!pip install polars
!pip install optuna optuna-integration

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math
import pandas as pd
import polars as pl
import numpy as np
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss,
    mean_absolute_error, mean_squared_error, r2_score
)
from xgboost import XGBRegressor, XGBClassifier
import optuna
from optuna.integration import XGBoostPruningCallback
import shap
import time
import joblib
import boto3
import tempfile
import logging

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(15)

In [ ]:
def save_model_to_s3(model, bucket_name, object_key):
    s3_client = boto3.client('s3')
    try:
        with tempfile.TemporaryFile() as fp:
            joblib.dump(model, fp)
            fp.seek(0)
            s3_client.put_object(Body=fp.read(), Bucket=bucket_name, Key=object_key)
            logging.info(f'{object_key} saved to s3 bucket {bucket_name}')
    except Exception as e:
        logging.exception(e)
        raise

In [ ]:
def load_model_from_s3(bucket_name, object_key):
    s3_client = boto3.client('s3')
    try:
        with tempfile.TemporaryFile() as fp:
            s3_client.download_fileobj(bucket_name, object_key, fp)
            fp.seek(0)
            model = joblib.load(fp)
            logging.info(f'{object_key} loaded from s3 bucket {bucket_name}')
            return model
    except Exception as e:
        logging.exception(e)
        raise

In [ ]:
# Read aggregated data from S3
df = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features_excluding_full_recovery_site.parquet")

In [ ]:
# Data Splitting
# Train on 2022-2024, test on 2025
df_train = df.filter(pl.col("year") < 2025)
df_test = df.filter(pl.col("year") == 2025)

In [ ]:
# Define features and target variable
identity_cols = [
    "hashed_fc", "gl_product_group", "week_date",
]

gl_composition_cols = [
    "share_food", "share_non_food", "share_pet_food",
    "share_RETAIL", "share_FBA", "share_hazmat",
]

gl_volume_cols = [
    "units_total", "cogs_total", "weight_total",
    "avg_cogs_per_unit", "avg_weight_per_unit", "cogs_per_unit_weight",
]

gl_at_site_cols = [
    "site_units_share_week", "site_weight_share_week",
]

site_context_cols = [
    "site_units_total_week", "site_weight_total_week",
    "site_type", "site_category", "country", "country_state",
]

temporal_site_context_cols = ['site_units_total_week_lag_1w',
 'site_units_total_week_lag_4w',
 'site_units_total_week_lag_12w',
 'site_units_total_week_lag_13w',
 'site_units_total_week_lag_52w',
 'site_weight_total_week_lag_1w',
 'site_weight_total_week_lag_4w',
 'site_weight_total_week_lag_12w',
 'site_weight_total_week_lag_13w',
 'site_weight_total_week_lag_52w',
 'site_prob_recovered_week_lag_1w',
 'site_prob_recovered_week_lag_4w',
 'site_prob_recovered_week_lag_12w',
 'site_prob_recovered_week_lag_13w',
 'site_prob_recovered_week_lag_52w',
 'site_prob_recovered_week_rolling_4w',
 'site_prob_recovered_week_rolling_12w',
'site_prob_recovered_week_rolling_26w',
 'site_prob_recovered_week_rolling_52w',
]

calendar_cols = [
    "month", "week"
]

temporal_composition_cols = [
 'share_RETAIL_lag_1w',
 'share_RETAIL_lag_4w',
 'share_RETAIL_lag_12w',
 'share_RETAIL_lag_13w',
 'share_RETAIL_lag_52w',
 'share_FBA_lag_4w',
 'share_FBA_lag_12w',
 'share_FBA_lag_13w',
 'share_FBA_lag_52w',
 'share_hazmat_lag_1w',
 'share_hazmat_lag_4w',
 'share_hazmat_lag_12w',
 'share_hazmat_lag_13w',
 'share_hazmat_lag_52w',
 'share_food_lag_1w',
 'share_food_lag_4w',
 'share_food_lag_12w',
 'share_food_lag_13w',
 'share_food_lag_52w',
 'share_non_food_lag_1w',
 'share_non_food_lag_4w',
 'share_non_food_lag_12w',
 'share_non_food_lag_13w',
 'share_non_food_lag_52w',
 'share_pet_food_lag_1w',
 'share_pet_food_lag_4w',
 'share_pet_food_lag_12w',
 'share_pet_food_lag_13w',
 'share_pet_food_lag_52w',
 'share_food_rolling_4w',
 'share_food_rolling_12w',
 'share_non_food_rolling_4w',
 'share_non_food_rolling_12w',
 'share_pet_food_rolling_4w',
 'share_pet_food_rolling_12w',
 'share_RETAIL_rolling_4w',
 'share_RETAIL_rolling_12w',
 'share_FBA_rolling_4w',
 'share_FBA_rolling_12w',
 'share_hazmat_rolling_4w',
 'share_hazmat_rolling_12w',
 'share_food_rolling_26w',
 'share_food_rolling_52w',
 'share_non_food_rolling_26w',
 'share_non_food_rolling_52w',
 'share_pet_food_rolling_26w',
 'share_pet_food_rolling_52w',
 'share_RETAIL_rolling_26w',
 'share_RETAIL_rolling_52w',
 'share_FBA_rolling_26w',
 'share_FBA_rolling_52w',
 'share_hazmat_rolling_26w',
 'share_hazmat_rolling_52w',
 'share_RETAIL_ewma_5a',
 'share_RETAIL_ewma_1a',
 'share_FBA_ewma_5a',
 'share_FBA_ewma_1a',
 'share_hazmat_ewma_5a',
 'share_hazmat_ewma_1a',
 'share_food_ewma_5a',
 'share_food_ewma_1a',
 'share_non_food_ewma_5a',
 'share_non_food_ewma_1a',
 'share_pet_food_ewma_5a',
 'share_pet_food_ewma_1a',
]

temporal_volume_cols = [
 'units_total_lag_1w',
 'units_total_lag_4w',
 'units_total_lag_12w',
 'units_total_lag_13w',
 'units_total_lag_52w',
 'cogs_total_lag_1w',
 'cogs_total_lag_4w',
 'cogs_total_lag_12w',
 'cogs_total_lag_13w',
 'cogs_total_lag_52w',
 'weight_total_lag_1w',
 'weight_total_lag_4w',
 'weight_total_lag_12w',
 'weight_total_lag_13w',
 'weight_total_lag_52w',
 'units_total_rolling_4w',
 'units_total_rolling_12w',
 'cogs_total_rolling_4w',
 'cogs_total_rolling_12w',
 'weight_total_rolling_4w',
 'weight_total_rolling_12w',
 'units_total_rolling_26w',
 'units_total_rolling_52w',
 'cogs_total_rolling_26w',
 'cogs_total_rolling_52w',
 'weight_total_rolling_26w',
 'weight_total_rolling_52w',
 'units_total_ewma_5a',
 'units_total_ewma_1a',
 'cogs_total_ewma_5a',
 'cogs_total_ewma_1a',
 'weight_total_ewma_5a',
 'weight_total_ewma_1a'
]

temporal_probability_cols = [
 'prob_recovered_lag_1w',
 'prob_recovered_lag_4w',
 'prob_recovered_lag_12w',
 'prob_recovered_lag_13w',
 'prob_recovered_lag_52w',
 'prob_recovered_rolling_26w',
 'prob_recovered_rolling_52w',
 'prob_recovered_rolling_4w',
 'prob_recovered_rolling_12w',
 'prob_recovered_ewma_5a',
 'prob_recovered_ewma_1a',
]

In [ ]:
feature_cols = (gl_composition_cols + gl_volume_cols + gl_at_site_cols
                + site_context_cols + temporal_site_context_cols + calendar_cols
                + temporal_composition_cols + temporal_volume_cols + temporal_probability_cols)

print(f"Total features: {len(feature_cols)}")

In [ ]:
df_train = df_train.with_columns(pl.when(pl.col('prob_recovered') > 0).then(1).otherwise(0).alias('binary_recovered'))
df_test = df_test.with_columns(pl.when(pl.col('prob_recovered') > 0).then(1).otherwise(0).alias('binary_recovered'))

In [ ]:
EPS = 1e-6  # boundary guard

def logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))

In [ ]:
# categorical columns that need dtype casting
CAT_COLS = [
    'hashed_fc',
    'gl_product_group',
    'country',
    'country_state',
    'site_type',
    'site_category',
]

def cast_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

In [ ]:
# Custom early-stopping eval metric for Stage 2 (operates in logit space,
# back-transforms before computing MAE in original probability space).
def prob_mae(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    p_true = sigmoid(y_true)
    p_pred = sigmoid(y_pred)
    return float(np.mean(np.abs(p_pred - p_true)))


def compute_weights_stage2(y: np.ndarray) -> np.ndarray:
    """Sample weights emphasizing the 10-60% rate buckets."""
    w = np.ones(len(y))
    w[(y >= 0.1) & (y < 0.3)] = 3.0
    w[(y >= 0.3) & (y < 0.6)] = 8.0
    w[(y >= 0.6)]              = 2.0
    return w

# Stage 1: Classifier Tuning

## Prepare Stage 1 data

Materialize the feature matrices once so each Optuna trial does not repeat the polars→pandas conversion.

In [ ]:
X_train_clf = cast_categoricals(df_train.select(feature_cols).to_pandas())
X_val_clf   = cast_categoricals(df_test.select(feature_cols).to_pandas())

y_train_clf = (df_train["prob_recovered"].to_pandas().values > 0).astype(np.float32)
y_val_clf   = (df_test["prob_recovered"].to_pandas().values > 0).astype(np.float32)

n_neg = (y_train_clf == 0).sum()
n_pos = (y_train_clf == 1).sum()
scale_pos_weight_clf = n_neg / n_pos
print(f"n_neg={n_neg:,}  n_pos={n_pos:,}  scale_pos_weight={scale_pos_weight_clf:.2f}")
print(f"X_train shape: {X_train_clf.shape}  X_val shape: {X_val_clf.shape}")

## Objective function

In [ ]:
def objective_stage1(trial: optuna.Trial) -> float:
    params = {
        "objective": "binary:logistic",
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "n_estimators": 500,            # lower ceiling; early stopping handles the rest
        "early_stopping_rounds": 30,    # tighter; stops faster on bad trials
        "eval_metric": ["aucpr", "logloss"],
        "scale_pos_weight": scale_pos_weight_clf,
        "callbacks": [XGBoostPruningCallback(trial, "validation_0-logloss")],
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_delta_step":   trial.suggest_int("max_delta_step", 0, 10),
    }

    model = XGBClassifier(**params)
    model.fit(
        X_train_clf, y_train_clf,
        eval_set=[(X_val_clf, y_val_clf)],
        verbose=False,
    )

    p_val = model.predict_proba(X_val_clf)[:, 1]
    # Minimize the negative of AUC-PR (threshold-independent, appropriate for imbalanced data).
    return -float(average_precision_score(y_val_clf, p_val))

## Run Stage 1 study

In [ ]:
study_clf = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=5),
    study_name="xgb_stage1_classifier",
)

study_clf.optimize(
    objective_stage1,
    n_trials=50,
    timeout=21600,  # 6 hours
    show_progress_bar=True,
)

print(f"\nBest AUC-PR: {-study_clf.best_value:.4f}")
print(f"Best params: {study_clf.best_params}")

In [ ]:
study_clf.best_params

## Retrain Stage 1 with tuned hyperparameters

In [ ]:
def train_classifier_tuned(
    df_train: pl.DataFrame,
    df_test: pl.DataFrame,
    feature_cols: list[str],
    best_params: dict,
    target_col: str = "prob_recovered",
) -> tuple:

    y_train = (df_train[target_col].to_pandas().values > 0).astype(np.float32)
    y_val   = (df_test[target_col].to_pandas().values > 0).astype(np.float32)

    X_train = cast_categoricals(df_train.select(feature_cols).to_pandas())
    X_val   = cast_categoricals(df_test.select(feature_cols).to_pandas())

    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    scale_pos_weight = n_neg / n_pos
    print(f"  n_neg={n_neg:,}  n_pos={n_pos:,}  scale_pos_weight={scale_pos_weight:.2f}")

    model_clf = XGBClassifier(
        objective="binary:logistic",
        n_estimators=2000,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        enable_categorical=True,
        random_state=42,
        early_stopping_rounds=50,
        eval_metric=["auc", "aucpr", "logloss"],
        **best_params,
    )

    model_clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100,
    )

    p_recovery = model_clf.predict_proba(X_val)[:, 1]

    print(f"\nStage 1 metrics (tuned):")
    print(f"  AUC-ROC:    {roc_auc_score(y_val, p_recovery):.4f}")
    print(f"  AUC-PR:     {average_precision_score(y_val, p_recovery):.4f}")
    print(f"  F1 (t=0.5): {f1_score(y_val, (p_recovery >= 0.5).astype(int)):.4f}")
    print(f"  Best iteration: {model_clf.best_iteration}")

    return model_clf, X_val, y_val, p_recovery

In [ ]:
start = time.perf_counter()
model_clf, X_val_clf_tuned, y_val_clf_tuned, p_nonzero = train_classifier_tuned(
    df_train, df_test, feature_cols, study_clf.best_params,
)
elapsed = (time.perf_counter() - start) / 60
print(f"\nTotal time: {elapsed:.1f} min")

In [ ]:
save_model_to_s3(model_clf, 'msds-26.2-data',
                 'model/tuned_xgboost_classification_model_excluding_full_recovery.joblib')

# Stage 2: Regressor Tuning

Stage 2 trains only on non-zero rows and uses sample weights emphasizing the 10-60% rate buckets. Target is logit-transformed; eval metric back-transforms to probability space before computing MAE.

## Prepare Stage 2 data

Filter to non-zero rows, build the `site_gl_baseline` features from training data, and materialize the tuning matrices once.

In [ ]:
# Hierarchical priors + fill helper for the three site-GL baseline features.
#
# site_gl_baseline is computed from df_train_nz and left-joined onto both
# train and test. Training rows always match (built from the same set), so
# XGBoost never sees NaN during training. Unseen (hashed_fc, gl_product_group)
# pairs at inference produce NaN, XGBoost routes NaN to an untrained default
# direction, and predictions get pushed toward recovery = 1.
#
# Fix: compute GL-level / site-level / global priors from df_train_nz, fill
# unseen combos hierarchically (GL -> site -> global), and during training
# hold out a small fraction of site-GL pairs so the model learns splits on
# filled-in values rather than routing blindly.
BASELINE_COLS = ["site_gl_mean_rate", "site_gl_std_rate", "site_gl_n_nonzero_weeks"]


def build_baseline_priors(df_train_nz: pl.DataFrame, target_col: str = "prob_recovered") -> dict:
    gl_baseline = (
        df_train_nz
        .group_by("gl_product_group")
        .agg([
            pl.col(target_col).mean().alias("gl_mean_rate"),
            pl.col(target_col).std().alias("gl_std_rate"),
        ])
    )
    site_baseline = (
        df_train_nz
        .group_by("hashed_fc")
        .agg([
            pl.col(target_col).mean().alias("site_mean_rate"),
            pl.col(target_col).std().alias("site_std_rate"),
        ])
    )
    return {
        "gl_baseline": gl_baseline,
        "site_baseline": site_baseline,
        "global_mean_rate": float(df_train_nz[target_col].mean()),
        "global_std_rate":  float(df_train_nz[target_col].std()),
    }


def fill_site_gl_baseline(df: pl.DataFrame, priors: dict) -> pl.DataFrame:
    """Fill NaNs in site_gl_mean_rate / site_gl_std_rate / site_gl_n_nonzero_weeks
    using GL -> site -> global fallbacks. n_nonzero_weeks is set to 0 on any
    fallback (honest zero support)."""
    df = df.join(priors["gl_baseline"],   on="gl_product_group", how="left")
    df = df.join(priors["site_baseline"], on="hashed_fc",        how="left")
    df = df.with_columns([
        pl.coalesce([
            pl.col("site_gl_mean_rate"),
            pl.col("gl_mean_rate"),
            pl.col("site_mean_rate"),
            pl.lit(priors["global_mean_rate"]),
        ]).alias("site_gl_mean_rate"),
        pl.coalesce([
            pl.col("site_gl_std_rate"),
            pl.col("gl_std_rate"),
            pl.col("site_std_rate"),
            pl.lit(priors["global_std_rate"]),
        ]).alias("site_gl_std_rate"),
        pl.col("site_gl_n_nonzero_weeks").fill_null(0).alias("site_gl_n_nonzero_weeks"),
    ])
    return df.drop(["gl_mean_rate", "gl_std_rate", "site_mean_rate", "site_std_rate"])


def holdout_site_gl_baseline(
    site_gl_baseline: pl.DataFrame,
    holdout_frac: float = 0.05,
    seed: int = 42,
) -> pl.DataFrame:
    """Return a copy of site_gl_baseline with `holdout_frac` of (hashed_fc, gl_product_group)
    pairs removed, so rows for those pairs fall through to the hierarchical fallback."""
    all_pairs = site_gl_baseline.select(["hashed_fc", "gl_product_group"])
    holdout_pairs = (
        all_pairs
        .sample(fraction=holdout_frac, seed=seed)
        .with_columns(pl.lit(True).alias("_held_out"))
    )
    return (
        site_gl_baseline
        .join(holdout_pairs, on=["hashed_fc", "gl_product_group"], how="left")
        .filter(pl.col("_held_out").is_null())
        .drop("_held_out")
    )


In [ ]:
target_col = "prob_recovered"

df_train_nz = df_train.filter(pl.col(target_col) > 0)
df_test_nz  = df_test.filter(pl.col(target_col) > 0)

print(f"Stage 2 - train rows: {len(df_train_nz):,}  test rows: {len(df_test_nz):,}")
print(f"Non-zero fraction - train: {len(df_train_nz)/len(df_train):.1%}  "
      f"test: {len(df_test_nz)/len(df_test):.1%}")

# Per site-GL recovery baseline features - computed from training data only.
site_gl_baseline = (
    df_train_nz
    .group_by(["hashed_fc", "gl_product_group"])
    .agg([
        pl.col(target_col).mean().alias("site_gl_mean_rate"),
        pl.col(target_col).std().alias("site_gl_std_rate"),
        pl.col(target_col).count().alias("site_gl_n_nonzero_weeks"),
    ])
)

# Hierarchical priors + train-time holdout so XGBoost learns splits on
# filled-in baseline values rather than routing NaN blindly.
priors = build_baseline_priors(df_train_nz, target_col=target_col)
HOLDOUT_FRAC = 0.05
site_gl_baseline_seen = holdout_site_gl_baseline(
    site_gl_baseline, holdout_frac=HOLDOUT_FRAC, seed=42,
)
print(f"  Holdout pairs (train-time unseen simulation): "
      f"{len(site_gl_baseline) - len(site_gl_baseline_seen):,} / {len(site_gl_baseline):,} "
      f"({HOLDOUT_FRAC:.1%})")

# Train joins the _seen subset so held-out rows get filled via priors.
df_train_nz = df_train_nz.join(
    site_gl_baseline_seen, on=["hashed_fc", "gl_product_group"], how="left",
)
df_train_nz = fill_site_gl_baseline(df_train_nz, priors)

# Test joins the full baseline; genuinely unseen combos get filled.
df_test_nz = df_test_nz.join(
    site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left",
)

extended_features = feature_cols + BASELINE_COLS

n_unseen = df_test_nz.filter(pl.col("site_gl_mean_rate").is_null()).height
print(f"\nSite-GL baseline coverage (pre-fill):")
print(f"  Test non-zero rows:       {len(df_test_nz):,}")
print(f"  Rows with unseen site-GL: {n_unseen:,} ({n_unseen/len(df_test_nz):.1%})")

df_test_nz = fill_site_gl_baseline(df_test_nz, priors)

X_train_reg = cast_categoricals(df_train_nz.select(extended_features).to_pandas())
X_val_reg   = cast_categoricals(df_test_nz.select(extended_features).to_pandas())

y_train_reg = df_train_nz[target_col].to_pandas().values
y_val_reg   = df_test_nz[target_col].to_pandas().values

y_train_logit = logit(y_train_reg)
y_val_logit   = logit(y_val_reg)

w_train_reg = compute_weights_stage2(y_train_reg)

print(f"\nX_train shape: {X_train_reg.shape}  X_val shape: {X_val_reg.shape}")


## Objective function

In [ ]:
def objective_stage2(trial: optuna.Trial) -> float:
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "n_estimators": 500,
        "early_stopping_rounds": 30,
        "eval_metric": prob_mae,
        "callbacks": [XGBoostPruningCallback(trial, "validation_0-prob_mae")],
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    model = XGBRegressor(**params)
    model.fit(
        X_train_reg, y_train_logit,
        sample_weight=w_train_reg,
        eval_set=[(X_val_reg, y_val_logit)],
        verbose=False,
    )

    preds = np.clip(sigmoid(model.predict(X_val_reg)), 0.0, 1.0)
    # MAE in original probability space (matches how Stage 2 is evaluated downstream).
    return float(mean_absolute_error(y_val_reg, preds))

## Run Stage 2 study

In [ ]:
study_reg = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=5),
    study_name="xgb_stage2_regressor",
)

study_reg.optimize(
    objective_stage2,
    n_trials=50,
    timeout=21600,  # 6 hours
    show_progress_bar=True,
)

print(f"\nBest MAE:    {study_reg.best_value:.4f}")
print(f"Best params: {study_reg.best_params}")

In [ ]:
study_reg.best_params

## Retrain Stage 2 with tuned hyperparameters

In [ ]:
def train_stage2_tuned(
    df_train: pl.DataFrame,
    df_test: pl.DataFrame,
    feature_cols: list[str],
    best_params: dict,
    target_col: str = "prob_recovered",
    holdout_frac: float = 0.05,
    holdout_seed: int = 42,
) -> tuple:

    df_train_nz = df_train.filter(pl.col(target_col) > 0)
    df_test_nz  = df_test.filter(pl.col(target_col) > 0)

    print(f"Stage 2 - train rows: {len(df_train_nz):,}  test rows: {len(df_test_nz):,}")

    site_gl_baseline = (
        df_train_nz
        .group_by(["hashed_fc", "gl_product_group"])
        .agg([
            pl.col(target_col).mean().alias("site_gl_mean_rate"),
            pl.col(target_col).std().alias("site_gl_std_rate"),
            pl.col(target_col).count().alias("site_gl_n_nonzero_weeks"),
        ])
    )

    # priors + holdout simulation of unseen combos during training
    priors = build_baseline_priors(df_train_nz, target_col=target_col)
    site_gl_baseline_seen = holdout_site_gl_baseline(
        site_gl_baseline, holdout_frac=holdout_frac, seed=holdout_seed,
    )
    print(f"  Holdout pairs: {len(site_gl_baseline) - len(site_gl_baseline_seen):,} / "
          f"{len(site_gl_baseline):,} ({holdout_frac:.1%})")

    df_train_nz = df_train_nz.join(
        site_gl_baseline_seen, on=["hashed_fc", "gl_product_group"], how="left",
    )
    df_train_nz = fill_site_gl_baseline(df_train_nz, priors)

    df_test_nz = df_test_nz.join(
        site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left",
    )

    extended_features = feature_cols + BASELINE_COLS

    n_unseen = df_test_nz.filter(pl.col("site_gl_mean_rate").is_null()).height
    print(f"  Rows with unseen site-GL (pre-fill): {n_unseen:,} ({n_unseen/len(df_test_nz):.1%})")

    df_test_nz = fill_site_gl_baseline(df_test_nz, priors)

    X_train = cast_categoricals(df_train_nz.select(extended_features).to_pandas())
    X_val   = cast_categoricals(df_test_nz.select(extended_features).to_pandas())

    y_train = df_train_nz[target_col].to_pandas().values
    y_val   = df_test_nz[target_col].to_pandas().values

    y_train_logit = logit(y_train)
    y_val_logit   = logit(y_val)

    model_reg = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=2000,
        tree_method="hist",
        enable_categorical=True,
        random_state=42,
        early_stopping_rounds=50,
        eval_metric=prob_mae,
        **best_params,
    )

    model_reg.fit(
        X_train, y_train_logit,
        sample_weight=compute_weights_stage2(y_train),
        eval_set=[(X_val, y_val_logit)],
        verbose=100,
    )

    preds = np.clip(sigmoid(model_reg.predict(X_val)), 0.0, 1.0)

    mae  = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2   = r2_score(y_val, preds)

    print(f"\nStage 2 metrics (tuned, non-zero rows only):")
    print(f"  MAE:            {mae:.4f}")
    print(f"  RMSE:           {rmse:.4f}")
    print(f"  R2:             {r2:.4f}")
    print(f"  Best iteration: {model_reg.best_iteration}")

    # persist the full baseline (not _seen) plus priors for inference
    model_reg.site_gl_baseline_ = site_gl_baseline
    model_reg.baseline_priors_  = priors
    return model_reg, X_val, y_val, preds, extended_features


In [ ]:
start = time.perf_counter()
model_reg, X_val_reg_tuned, y_val_reg_tuned, preds_nz, extended_features = train_stage2_tuned(
    df_train, df_test, feature_cols, study_reg.best_params,
)
elapsed = (time.perf_counter() - start) / 60
print(f"\nTotal time: {elapsed:.1f} min")

In [ ]:
save_model_to_s3(model_reg, 'msds-26.2-data',
                 'model/tuned_xgboost_regression_model_excluding_full_recovery.joblib')

# Combined Two-Stage Evaluation

Product of the two tuned stages: `final_pred = p_nonzero * e_rate`.

In [ ]:
# Load clf model if not trained
model_clf = load_model_from_s3(
    "msds-26.2-data", "model/tuned_xgboost_classification_model_excluding_full_recovery.joblib"
)

In [ ]:
def predict_two_stage(
    model_clf,
    model_reg,
    df_test: pl.DataFrame,
    feature_cols: list[str],
    extended_features: list[str],
    site_gl_baseline: pl.DataFrame,
    target_col: str = "prob_recovered",
    priors: dict | None = None,
) -> tuple:

    y_val = df_test[target_col].to_pandas().values

    X_val_clf = cast_categoricals(df_test.select(feature_cols).to_pandas())
    p_nonzero = model_clf.predict_proba(X_val_clf)[:, 1]

    df_test_ext = df_test.join(
        site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left",
    )
    if priors is None:
        priors = getattr(model_reg, "baseline_priors_", None)
    if priors is not None:
        df_test_ext = fill_site_gl_baseline(df_test_ext, priors)

    X_val_reg = cast_categoricals(df_test_ext.select(extended_features).to_pandas())
    e_rate    = np.clip(sigmoid(model_reg.predict(X_val_reg)), 0.0, 1.0)

    preds = p_nonzero * e_rate

    mae  = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2   = r2_score(y_val, preds)

    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R2:   {r2:.4f}")

    return preds, y_val, mae, rmse, r2


In [ ]:
preds_two_stage, y_val, mae, rmse, r2 = predict_two_stage(
    model_clf, model_reg,
    df_test,
    feature_cols,
    extended_features,
    model_reg.site_gl_baseline_,
)

# Optuna study inspection

Top trials for each study, for sanity-checking the tuning trajectory.

In [ ]:
print("Stage 1 - top 10 trials by objective (lower = better, value = -AUC-PR):")
study_clf.trials_dataframe().sort_values("value").head(10)

In [ ]:
print("Stage 2 - top 10 trials by objective (lower = better, value = MAE):")
study_reg.trials_dataframe().sort_values("value").head(10)